In [2]:
import os
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession

PROJECT_ROOT = os.path.dirname(os.getcwd())
# clickstream events replayer
# TARGET_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "bronze_clickstream")

# kafka udp events
TARGET_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "kafka_bronze_clickstream")

builder = (
    SparkSession.builder.appName("Clickstream_Bronze_Query")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

# %%
df = spark.read.format("delta").load(TARGET_PATH)
df.show(5, truncate=False)

:: loading settings :: url = jar:file:/Users/intuitivecode/Code/deltastreams/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/intuitivecode/.ivy2/cache
The jars for the packages stored in: /Users/intuitivecode/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a39e09d2-2e54-4c90-af4f-9ac314d1b570;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 85ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default    

+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+---------+------+-----------------------+
|kafka_key|value                                                                                                                                                                                                             |topic            |partition|offset|kafka_timestamp        |
+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+---------+------+-----------------------+
|NULL     |{"host":"172.20.0.1","message":"2026-10-16T15:45:00Z firewall-core-01 SRC=192.168.1.100 DST=10.0.0.5 PROTO=TCP SPT=443 DPT=5140","port":38423,"

In [4]:
# Load Delta Table Reference and Record Count

df = spark.read.format("delta").load(TARGET_PATH)
delta_table = DeltaTable.forPath(spark, TARGET_PATH)

print(f"Total Bronze Records Ingested: {df.count():,}")

Total Bronze Records Ingested: 711


26/10/17 14:26:49 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
# View Schema And Sample Records (Clickstream events replayer)

df.printSchema()
df.select("event_time", "event_type", "price", "_ingested_at", "_source_file").show(10, truncate=False)

In [ ]:
# Check Delta Commit History
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

In [ ]:
# Inspect Corrupt Payloads (DLQ Candidate Check)

from pyspark.sql.functions import col

# Inspect Corrupt Payloads (DLQ Candidate Check)
corrupt_df = df.filter(col("price") == "CORRUPT_PRICE_PAYLOAD")
print(f"Corrupt Records Count: {corrupt_df.count()}")
corrupt_df.select("event_time", "event_type", "price", "_source_file").show(10, truncate=False)

